# Disaster Recovery Cost Prediction and Resilience Optimization
## Week 2 Day 1 — Feature Engineering

## Goal of this notebook

The purpose of this notebook is to create a processed disaster-level dataset suitable for modelling.

This includes:
- merging the three FEMA datasets
- aggregating project-level Public Assistance data to disaster level
- engineering temporal, geographic, historical, and project-level features
- creating a log-transformed version of the target
- saving the final processed dataset

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.processing.feature_engineering import build_processed_dataset, inspect_processed_dataset

processed_df = build_processed_dataset()
inspect_processed_dataset(processed_df)

2026-04-09 08:21:45,486 | INFO | src.processing.feature_engineering | Loaded declarations | shape=(69770, 28)
2026-04-09 08:21:45,499 | INFO | src.processing.feature_engineering | Loaded public assistance | shape=(810787, 25)
2026-04-09 08:21:45,499 | INFO | src.processing.feature_engineering | Loaded web summaries | shape=(3909, 14)
2026-04-09 08:21:45,962 | INFO | src.processing.feature_engineering | Clipping 923 negative values in column 'projectAmount' to zero
2026-04-09 08:21:45,981 | INFO | src.processing.feature_engineering | Clipping 923 negative values in column 'federalShareObligated' to zero
2026-04-09 08:21:45,998 | INFO | src.processing.feature_engineering | Clipping 946 negative values in column 'totalObligated' to zero
2026-04-09 08:21:46,111 | INFO | src.processing.feature_engineering | Clipping 1 negative values in column 'totalObligatedAmountPa' to zero
2026-04-09 08:21:46,113 | INFO | src.processing.feature_engineering | Clipping 2 negative values in column 'totalObl


Processed dataset shape: (5163, 43)

Processed dataset columns:
- disasterNumber
- state
- incidentType
- declarationType
- declarationDate
- incidentBeginDate
- incidentEndDate
- declaration_year
- declaration_month
- incident_duration_days
- season
- census_region
- state_5yr_disaster_count
- high_cost_incident
- project_count
- avg_project_amount
- total_obligated_amount
- log_total_obligated_amount
- fyDeclared
- ihProgramDeclared
- iaProgramDeclared
- paProgramDeclared
- hmProgramDeclared
- tribalRequest
- fipsStateCode
- fipsCountyCode
- placeCode
- declarationRequestNumber
- incidentId
- region
- femaDeclarationString
- declarationTitle
- disasterCloseoutDate
- designatedArea
- lastIAFilingDate
- designatedIncidentTypes
- lastRefresh
- hash
- id
- totalObligatedAmountPa
- totalObligatedAmountCatAb
- totalObligatedAmountCatC2g
- totalObligatedAmountHmgp

Sample rows:
   disasterNumber state incidentType declarationType           declarationDate         incidentBeginDate         

## Notes on engineered features

### Temporal features
- `declaration_year`
- `declaration_month`
- `incident_duration_days`

These features capture timing and duration effects that may influence recovery cost.

### Seasonality feature
- `season`

This groups declaration months into broader seasonal categories.

### Geographic feature
- `census_region`

This provides a regional view of hazard exposure and cost patterns.

### Historical frequency
- `state_5yr_disaster_count`

This approximates recent disaster activity within each state.

### Risk flag
- `high_cost_incident`

This is a simple boolean indicator for incident types that often carry large recovery costs.

### Project-level aggregates
- `project_count`
- `avg_project_amount`

These summarize the Public Assistance project structure for each disaster.

### Target transformation
- `log_total_obligated_amount`

This helps manage skewness in disaster recovery cost.

## Data Quality Checks After Feature Engineering

After building the processed dataset, additional checks were performed to ensure consistency:

- Missing `incidentType` values were filled with "Unknown" to avoid issues in categorical modelling.
- Missing `incident_duration_days` values were set to 0 where duration could not be computed.
- Public Assistance aggregates such as `project_count` and `total_obligated_amount` naturally contain zeros for disasters without recorded funding.

These adjustments ensure the dataset is consistent and ready for feature analysis and modelling.

## Week 2 Day 1 — Feature Engineering

In this stage, I developed a full feature engineering pipeline to transform validated FEMA datasets into a disaster-level modelling dataset.

The pipeline merges disaster declarations, Public Assistance project data, and FEMA summary data using `disasterNumber` as the key identifier. Project-level Public Assistance records were aggregated to create the target variable `total_obligated_amount`, along with supporting features such as project count and average project size.

I engineered temporal features including declaration year, month, and incident duration, and created a seasonality feature to capture cyclical disaster patterns. Geographic information was incorporated by mapping states to US Census regions. A rolling five-year disaster frequency feature was added to represent historical exposure, and a high-risk incident flag was created for disaster types commonly associated with large recovery costs.

The target variable was log-transformed to address skewness identified during exploratory data analysis. Additional data consistency checks were applied to handle missing categorical values and incomplete duration calculations.

The final dataset was saved as `processed_disasters.csv`, containing 69,770 disaster records and 43 features, providing a robust foundation for feature analysis and model development.